In [ ]:
%load_ext autoreload
%autoreload 2

## 1. Imports

In [ ]:
import os
import sys

sys.path.append("..")

import random

import numpy as np
import torch
from tqdm import tqdm

import wandb
from src.models.light_gcot import LightGCOT
from src.samplers.from_dataset import DatasetSampler
from src.samplers.primary import StandardNormalSampler, SwissRollSampler
from src.utils.discrete_ot import OTPlanSampler
from src.utils.paired import generate_paired_data, get_GT_points, get_paired_sampler
from src.utils.plotting.distributions import plot_swiss_roll
from src.utils.plotting.parameters import (
    plot_A_parameters,
    plot_B_parameters,
    plot_Z_parameters,
)
from src.utils.train import compute_loss, update_average

In [ ]:
device = torch.device(f"cuda:{torch.cuda.current_device()}" if torch.cuda.is_available() else "cpu")
device

In [ ]:
torch.set_default_device(device)
dtype = torch.float64
torch.torch.set_default_dtype(dtype)

## 2. Config

In [ ]:
X_DIM = 2
Y_DIM = 2
assert X_DIM > 1
assert Y_DIM > 1

OUTPUT_SEED = 42

N_POTENTIALS = 50
M_POTENTIALS = 10
EPSILON = 1.0
INIT_BY_SAMPLES = True
A_DIAGONAL_INIT = 0.1

BATCH_SIZE = 128
SAMPLING_BATCH_SIZE = 128

D_LR_PAIRED = 3e-4  # 1e-3 for eps 0.1, 0.01 and 3e-4 for eps 0.002
D_LR_UNPAIRED = 1e-3 
D_GRADIENT_MAX_NORM = float("inf")

M_X_UNPAIRED_SAMPLES = 1024
N_Y_UNPAIRED_SAMPLES = 1024
L_PAIRED_SAMPLES = 64

PLOT_EVERY = 1000
MAX_STEPS = 250000
CONTINUE = -1

EXP_COST = "MLP"
EXP_COST_INCLUDED = True
MINIBATCH_COST = "rotation-v2"

EMA_UPDATE = False

In [ ]:
torch.manual_seed(OUTPUT_SEED)
np.random.seed(OUTPUT_SEED)

In [ ]:
EXP_META_INFO = ""
EXP_NAME = (
    "Light-GCOT_Swiss_Roll_"
    + f"EPSILON_{EPSILON}_"
    + f"MAX_STEPS_{MAX_STEPS}_"
    + f"N_{N_POTENTIALS}_"
    + f"M_{M_POTENTIALS}_"
    + f"with_{EXP_COST}_cost_included_{EXP_COST_INCLUDED}_"
    + f"M_X_UNPAIRED_{M_X_UNPAIRED_SAMPLES}_"
    + f"N_Y_UNPAIRED_{N_Y_UNPAIRED_SAMPLES}_"
    + f"L_PAIRED_{L_PAIRED_SAMPLES}_"
    + f"LR_PAIRED_{D_LR_PAIRED}_"
    + f"LR_UNPAIRED_{D_LR_UNPAIRED}_"
    + f"MINIBATCH_COST_{MINIBATCH_COST}_"
    + EXP_META_INFO
)
OUTPUT_PATH = "../checkpoints/{}".format(EXP_NAME)

config = dict(
    X_DIM=X_DIM,
    Y_DIM=Y_DIM,
    D_LR_PAIRED=D_LR_PAIRED,
    D_LR_UNPAIRED=D_LR_UNPAIRED,
    BATCH_SIZE=BATCH_SIZE,
    EPSILON=EPSILON,
    D_GRADIENT_MAX_NORM=D_GRADIENT_MAX_NORM,
    N_POTENTIALS=N_POTENTIALS,
    M_POTENTIALS=M_POTENTIALS,
    INIT_BY_SAMPLES=INIT_BY_SAMPLES,
    A_DIAGONAL_INIT=A_DIAGONAL_INIT,
    M_X_UNPAIRED_SAMPLES=M_X_UNPAIRED_SAMPLES,
    N_Y_UNPAIRED_SAMPELS=N_Y_UNPAIRED_SAMPLES,
    L_PAIRED_SAMPLES=L_PAIRED_SAMPLES,
)

if not os.path.exists(OUTPUT_PATH):
    os.makedirs(OUTPUT_PATH)

## 3. Create data and samplers

In [ ]:
X_sampler = StandardNormalSampler(dim=2, device=device)
Y_sampler = SwissRollSampler(dim=2, device=device, dtype=dtype)

In [ ]:
otp_sampler = OTPlanSampler("sinkhorn", cost_function=MINIBATCH_COST)

In [ ]:
data_dir = "checkpoints/Tensors"
file_postfix = f"{MINIBATCH_COST}_{L_PAIRED_SAMPLES}"

In [ ]:
X_paired_train, Y_paired_train, X_paired_test, Y_paired_test = generate_paired_data(
    X_sampler, Y_sampler, otp_sampler, L_PAIRED_SAMPLES, "./checkpoints/Tensors", file_postfix, device=device
)

In [ ]:
pd_train_sampler = get_paired_sampler(X_paired_train, Y_paired_train, BATCH_SIZE, L_PAIRED_SAMPLES, device)

In [ ]:
X_unpaired_test = X_sampler.sample(L_PAIRED_SAMPLES)
Y_unpaired_test = Y_sampler.sample(L_PAIRED_SAMPLES)

In [ ]:
if M_X_UNPAIRED_SAMPLES > 0:
    source_data = X_sampler.sample(M_X_UNPAIRED_SAMPLES)
    usd_sampler = DatasetSampler(source_data, device=device) # usd - unpaired source data
else:
    usd_sampler = DatasetSampler(X_paired_train, device=device)

if N_Y_UNPAIRED_SAMPLES > 0:
    target_data = Y_sampler.sample(N_Y_UNPAIRED_SAMPLES)
    utd_sampler = DatasetSampler(target_data, device=device) # utd - unpaired target data
else:
    utd_sampler = DatasetSampler(Y_paired_train, device=device)

## 4. Model initialization

In [ ]:
D = LightGCOT(
    x_dim=X_DIM,
    y_dim=Y_DIM,
    n_potentials=N_POTENTIALS,
    m_potentials=M_POTENTIALS,
    epsilon=EPSILON,
    sampling_batch_size=SAMPLING_BATCH_SIZE,
    A_diagonal_init=A_DIAGONAL_INIT,
    cost_function=EXP_COST,
).to(dtype)

if INIT_BY_SAMPLES:
    D.init_a_by_samples(Y_sampler.sample(N_POTENTIALS))

In [ ]:
unpaired_params_to_update = [{"params": [D.log_w_n, D.a_n, D.log_A_n]}]

D_opt_unpaired = torch.optim.Adam(unpaired_params_to_update, lr=D_LR_UNPAIRED)

In [ ]:
# Change if cost is not MLP
if EXP_COST == "parameters":
    paired_params_to_update = [{"params": [D.log_v_m, D.b_m]}]
elif EXP_COST == "parameters_with_MLP":
    paired_params_to_update = [{"params": [D.log_v_m, D.b_m[0].weight, D.b_m[0].bias]}]
elif EXP_COST == "MLP":
    paired_params_to_update = [
        {"params": [D.log_v_m[0][0].weight, D.log_v_m[0][0].bias, D.b_m[0].weight, D.b_m[0].bias]}
    ]
elif EXP_COST == "MLP_deep":
    paired_params_to_update = [
        {
            "params": [
                D.log_v_m[0][0].weight,
                D.log_v_m[0][0].bias,
                D.b_m[0].weight,
                D.b_m[0].bias,
                D.b_m[3].weight,
                D.b_m[3].bias,
            ]
        }
    ]
else:
    raise ValueError("Unkown cost!")

D_opt_paired = torch.optim.Adam(paired_params_to_update, lr=D_LR_PAIRED, weight_decay=0.01)

In [ ]:
if CONTINUE > -1:
    D_opt_unpaired.load_state_dict(torch.load(os.path.join(OUTPUT_PATH, f"D_opt_unpaired_{CONTINUE}.pt")))
    D_opt_paired.load_state_dict(torch.load(os.path.join(OUTPUT_PATH, f"D_opt_paired_{CONTINUE}.pt")))

In [ ]:
# For EMA update
if EMA_UPDATE:
    D_copy = LightGCOT(
        x_dim=X_DIM,
        y_dim=Y_DIM,
        n_potentials=N_POTENTIALS,
        m_potentials=M_POTENTIALS,
        epsilon=EPSILON,
        sampling_batch_size=SAMPLING_BATCH_SIZE,
        A_diagonal_init=A_DIAGONAL_INIT,
        cost_function=EXP_COST,
    ).to(dtype)

## 5. Model training

In [ ]:
starting_points = torch.tensor([[-2.0, 0.0], [0.0, 0.0], [0.0, -2.0]])
num_ending_points = 64

In [ ]:
num_starting_points_paired = 5
indices = random.choices(range(L_PAIRED_SAMPLES), k=num_starting_points_paired)
starting_points_paired = X_paired_train[indices]
ending_points_paired = Y_paired_train[indices]

In [ ]:
gt_Y_points = get_GT_points(X_sampler, Y_sampler, otp_sampler, starting_points)

In [ ]:
wandb.init(name=EXP_NAME, config=config)

for step in tqdm(range(CONTINUE + 1, MAX_STEPS)):
    # training loop
    D_opt_unpaired.zero_grad()

    X = usd_sampler.sample(BATCH_SIZE)
    Y = utd_sampler.sample(BATCH_SIZE)

    log_v_m = D.compute_log_v_m(X)  # [bs x M]
    b_m = D.compute_b_m(X)  # [bs x M x y_dim]

    log_w_n = D.compute_log_w_n()  # [N]
    a_n = D.compute_a_n()  # [N x y_dim]
    A_n = D.compute_A_n()  # [N x y_dim]

    f_c = D.compute_dual_potential(log_w_n, a_n, A_n, log_v_m, b_m)
    f = D.compute_primal_potential(Y, log_w_n, a_n, A_n)

    D_loss_unpaired = -(f_c + f).mean()
    
    wandb.log({f"D unpaired loss": D_loss_unpaired.item()}, step=step)

    if EXP_COST_INCLUDED:
        D_opt_paired.zero_grad()
        X_paired, Y_paired = pd_train_sampler.sample(BATCH_SIZE)
        log_v_m_paired = D.compute_log_v_m(X_paired)  # [bs x M]
        b_m_paired = D.compute_b_m(X_paired)  # [bs x M x y_dim]

        c = D.compute_cost(Y_paired, log_v_m_paired, b_m_paired)
        D_loss_paired = c.mean()
        
        wandb.log({r"$c(x, y)$": c.mean().item()}, step=step)
        wandb.log({f"D paired loss": D_loss_paired.item()}, step=step)
        
    D_loss = D_loss_unpaired + D_loss_paired
    D_loss.backward()
    D_opt_paired.step(); D_opt_unpaired.step()
    
    D_unpaired_gradient_norm = torch.nn.utils.clip_grad_norm_(
        unpaired_params_to_update[0]["params"], max_norm=D_GRADIENT_MAX_NORM
    )
    D_paired_gradient_norm = torch.nn.utils.clip_grad_norm_(
        paired_params_to_update[0]["params"], max_norm=D_GRADIENT_MAX_NORM
    )
    D_gradient_norm = torch.nn.utils.clip_grad_norm_(D.parameters(), max_norm=D_GRADIENT_MAX_NORM)
    wandb.log({f"D unpaired gradient norm": D_unpaired_gradient_norm.item()}, step=step)
    wandb.log({f"D paired gradient norm": D_paired_gradient_norm.item()}, step=step)
    wandb.log({f"D gradient norm": D_gradient_norm.item()}, step=step)

    if EMA_UPDATE:
        update_average(D_copy, D, 0.99)
        model = D_copy
    else:
        model = D

    wandb.log({f"D loss": D_loss}, step=step)
    wandb.log(
        {f"Train paired loss": compute_loss(model, X_paired_train, Y_paired_train, X_paired_train, Y_paired_train).item()},
        step=step,
    )
    wandb.log(
        {f"Test paired loss": compute_loss(model, X_paired_test, Y_paired_test, X_paired_test, Y_paired_test).item()},
        step=step,
    )
    wandb.log(
        {f"Test unpaired loss": compute_loss(model, X_unpaired_test, Y_unpaired_test, X_paired_test, Y_paired_test).item()},
        step=step,
    )

    wandb.log({r"$-f^c(x)$": -f_c.mean().item()}, step=step)
    wandb.log({r"$-f(y)$": -f.mean().item()}, step=step)
    wandb.log({r"$-f(y)-f^c(x)$": -(f_c + f).mean().item()}, step=step)
    wandb.log({f"lam_min(A_n)": torch.min(A_n)}, step=step)
    wandb.log({f"lam_max(A_n)": torch.max(A_n)}, step=step)

    if step % PLOT_EVERY == 0:
        A_dict = plot_A_parameters(model, log=True)
        B_dict = plot_B_parameters(model, starting_points, log=True)
        if num_starting_points_paired > 0:
            Z_dict = plot_Z_parameters(model, starting_points, starting_points_paired, ending_points_paired, log=True)
        else:
            Z_dict = plot_Z_parameters(model, starting_points, log=True)
        distr_dict = plot_swiss_roll(
            {f"M={M_X_UNPAIRED_SAMPLES}, N={N_Y_UNPAIRED_SAMPLES}, L={L_PAIRED_SAMPLES}": model},
            X_sampler,
            Y_sampler,
            X_paired,
            Y_paired,
            starting_points,
            gt_Y_points,
            log=True,
        )
        wandb.log(A_dict | B_dict | Z_dict | distr_dict)

        torch.save(D.state_dict(), os.path.join(OUTPUT_PATH, f"D_{step}.pt"))
        
torch.save(D.state_dict(), os.path.join(OUTPUT_PATH, f"D_{MAX_STEPS}.pt"))
torch.save(D_opt_paired.state_dict(), os.path.join(OUTPUT_PATH, f"D_opt_paired_{MAX_STEPS}.pt"))
torch.save(D_opt_unpaired.state_dict(), os.path.join(OUTPUT_PATH, f"D_opt_unpaired_{MAX_STEPS}.pt"))

wandb.finish()

## Plotting

In [ ]:
M_X_unpaired_samples_list = [0, 1024]
N_Y_unpaired_samples_list = [0, 1024]
log_step = 99000

In [ ]:
models_dict = dict()

for i, M_X_unpaired_samples in enumerate(M_X_unpaired_samples_list):
    for j, N_Y_unpaired_samples in enumerate(N_Y_unpaired_samples_list):
        model = LightGCOT(
            x_dim=X_DIM,
            y_dim=Y_DIM,
            n_potentials=N_POTENTIALS,
            m_potentials=M_POTENTIALS,
            epsilon=EPSILON,
            sampling_batch_size=SAMPLING_BATCH_SIZE,
            A_diagonal_init=A_DIAGONAL_INIT,
            cost_function=EXP_COST,
        )
        exp_name = EXP_NAME.replace(
            f"M_X_UNPAIRED_{M_X_UNPAIRED_SAMPLES}_N_Y_UNPAIRED_{N_Y_UNPAIRED_SAMPLES}_",
            f"M_X_UNPAIRED_{M_X_unpaired_samples}_N_Y_UNPAIRED_{N_Y_unpaired_samples}_",
        )
        print(exp_name)
        output_path = "../checkpoints/{}".format(exp_name)
        model.load_state_dict(torch.load(os.path.join(output_path, f"D_{log_step}.pt"), map_location=device))
        title = f"M={M_X_unpaired_samples}, N={N_Y_unpaired_samples}, L={L_PAIRED_SAMPLES}"
        models_dict[title] = model

In [ ]:
plot_swiss_roll(
    models_dict,
    X_sampler,
    Y_sampler,
    X_paired_train,
    Y_paired_train,
    starting_points,
    gt_Y_points,
) 